In [2]:
import pandas as pd
import sqlite3



In [3]:
conn = sqlite3.connect(':memory:') # Create temporary database
items= pd.read_csv('C:\\Users\\Harry\\OneDrive\\Desktop\\freshflow\\freshflow\\data\\items.csv').to_sql('items', conn, index=False)

In [4]:
QUERY = "SELECT * FROM items"
pd.read_sql_query(QUERY, conn)

,item_id,description,dept,category,unit_of_measure,case_size,unit_cost
0,10001,Bananas,Produce,Tropical,LB,36,0.28
1,10002,Strawberries 1lb,Produce,Berries,LB,20,2.32
2,10003,Blueberries 6oz,Produce,Berries,LB,18,2.84
3,10004,Romaine Hearts 3ct,Produce,Salad,EA,18,3.34
4,10005,Baby Spinach 5oz,Produce,Salad,LB,24,1.93
...,...,...,...,...,...,...,...
195,10196,Coffee Ground 12oz,Grocery,Beverages,EA,48,0.81
196,10197,Tea Bags 40ct,Grocery,Beverages,EA,15,5.90
197,10198,Soda Cola 12pk,Grocery,Beverages,EA,30,4.39
198,10199,Sparkling Water 12pk,Grocery,Beverages,EA,40,2.44


In [5]:
sales_daily = pd.read_csv('C:\\Users\\Harry\\OneDrive\\Desktop\\freshflow\\freshflow\\data\\sales_daily.csv').to_sql('sales_daily', conn, index=False)
shipments = pd.read_csv('C:\\Users\\Harry\\OneDrive\\Desktop\\freshflow\\freshflow\\data\\shipments.csv').to_sql('shipments', conn, index=False)

In [10]:

conn.execute( '''          create view shrink_by_item_store_date as
select 
items.item_id,
shipments.store_id,
items.dept,
items.category,
shipments.date,
sum(shipments.cases_received*items.case_size) as units_shipped_on_date,
sum(sales_daily.units_sold) as units_sold_on_date,
sum(shipments.cases_received*items.case_size)- sum(sales_daily.units_sold) as shrink_on_date
from items
left join shipments on items.item_id = shipments.item_id
left join(
    select
    item_id,
     store_id,
     date,
     sum(units_sold) as units_sold
    from sales_daily
    group by item_id, store_id, date
)sales_daily on items.item_id = sales_daily.item_id and shipments.store_id = sales_daily.store_id and shipments.date = sales_daily.date
group by items.item_id, shipments.store_id, items.dept, items.category, shipments.date

''')

OperationalError: view shrink_by_item_store_date already exists

In [9]:
pd.read_sql_query( '''

create view shrink_by_item_store_date as
select 
items.item_id,
shipments.store_id,
items.dept,
items.category,
shipments.date,
sum(shipments.cases_received*items.case_size) as units_shipped_on_date,
sum(sales_daily.units_sold) as units_sold_on_date,
sum(shipments.cases_received*items.case_size)- sum(sales_daily.units_sold) as shrink_on_date
from items
left join shipments on items.item_id = shipments.item_id
left join(
    select
    item_id,
     store_id,
     date,
     sum(units_sold) as units_sold
    from sales_daily
    group by item_id, store_id, date
)sales_daily on items.item_id = sales_daily.item_id and shipments.store_id = sales_daily.store_id and shipments.date = sales_daily.date
group by items.item_id, shipments.store_id, items.dept, items.category, shipments.date

''',conn)

TypeError: 'NoneType' object is not iterable

In [12]:
pd.read_sql_query( '''
select distinct * from shrink_by_item_store_date
''',conn)

,item_id,store_id,dept,category,date,units_shipped_on_date,units_sold_on_date,shrink_on_date
0,10001,101,Produce,Tropical,2026-04-01,144,51.0,93.0
1,10001,101,Produce,Tropical,2026-04-04,144,40.0,104.0
2,10001,101,Produce,Tropical,2026-04-07,108,50.0,58.0
3,10001,101,Produce,Tropical,2026-04-10,180,14.0,166.0
4,10001,101,Produce,Tropical,2026-04-13,108,55.0,53.0
...,...,...,...,...,...,...,...,...
42545,10200,110,Grocery,Household,2026-05-13,45,3.0,42.0
42546,10200,110,Grocery,Household,2026-05-23,45,5.0,40.0
42547,10200,110,Grocery,Household,2026-06-02,45,6.0,39.0
42548,10200,110,Grocery,Household,2026-06-12,45,4.0,41.0


In [14]:
pd.read_sql_query( '''
select * from sales_daily
where item_id = 10001 and store_id = 101	
and date = '2026-04-01'
''',conn)

,date,store_id,banner,region,item_id,units_sold,net_sales
0,2026-04-01,101,Meridian Foods,Northeast,10001,51,20.91
